# AI agent with MCPs using vLLM and PydanticAI

**Author**: Mahdi Ghodsi  
**Knowledge level**: Beginner

This tutorial leverages AMD GPUs and **Model Context Protocol (MCP)**, an open standard for exposing LLM tools using an API, to deploy powerful language models like Qwen3. The key components for this tutorial are:

- 🖥️ **vLLM** for GPU-optimized inference
- 🛠️ **PydanticAI** for agent and tool management
- 🔌 **MCP Servers** for prebuilt tool integration

You'll learn how to set up your environment, deploy large language models like Qwen3, connect them to real-world tools using MCP, and build a conversational agent capable of reasoning and taking actions.

By the end of this workshop, you’ll have built an AI-powered Airbnb assistant agent that can find a place to stay based on preferences like location, budget, and travel dates.

This tutorial includes the following sections:

- [1. Launching the vLLM server](#step1)
- [2. Installing dependencies](#step2)
- [3. Create a simple instance of a PydanticAI agent](#step3)
- [4. Write a date/time tool for your agent](#step4)
- [5. Replace the date/time tool with an MCP server](#step5)
- [6. Turn your agent into an Airbnb finder](#step6)
- [7. Challenge to expand the agent](#step7)


## Prerequisites

This tutorial was developed and tested using the following setup. 

### Operating system

* **Ubuntu 22.04/24.04**: Ensure your system is running Ubuntu version 22.04 or 24.04.

### Hardware

* **AMD Instinct™ GPUs**: This tutorial was tested on an AMD Instinct MI300X GPU. Ensure you are using an AMD Instinct GPU or compatible hardware with ROCm support and that your system meets the [official requirements](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html).

### Software

* **ROCm**: The `vllm/vllm-openai-rocm:v0.31.0` container used below includes its own ROCm libraries; install a ROCm release that supports your GPU on the host. Install and verify ROCm by following the [ROCm install guide](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html). After installation, confirm your setup using:

    ``` bash
    amd-smi
    ```

    This command lists your AMD GPUs with relevant details.
    
    **Note**: For ROCm 6.4 and earlier, use the `rocm-smi` command instead.

<a id="step1"></a>

## Step 1: Launching the vLLM server

In this workshop, you'll use [vLLM](https://github.com/vllm-project/vllm) as your inference serving engine. vLLM provides many benefits, such as fast model execution, an extensive list of supported models, and ease of use. Best of all, it's open source. 

### Deploy the Qwen3-30B-A3B model with vLLM




It's time to start your vLLM server and create an endpoint for your LLM. The cells below start the vLLM server in the background and wait until it is ready.



The tutorial runs its servers from the notebook. These helpers start a server in the background, wait until it answers (or stop with its log if it fails), and stop it again:

In [1]:
import os
import signal
import subprocess
import time
import urllib.request


def server_is_up(url, headers=None):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}),
                                    timeout=5):
            return True
    except Exception:
        return False


def ensure_server(command, url, log_path, timeout=1200, headers=None):
    """Start `command` in the background (its output goes to log_path) unless
    url already answers, then wait until it does. Returns the server process,
    or None if a server was already running. Raises, with the end of the log,
    if the server exits or is not ready within timeout seconds."""
    if server_is_up(url, headers):
        print(f"Already running: {url}")
        return None
    log = open(log_path, "w")
    server = subprocess.Popen(command, shell=True, stdout=log, stderr=subprocess.STDOUT,
                              start_new_session=True)
    deadline = time.time() + timeout
    while time.time() < deadline and server.poll() is None:
        if server_is_up(url, headers):
            print(f"Ready after {timeout - int(deadline - time.time())}s: {url}")
            return server
        time.sleep(5)
    tail = "".join(open(log_path).readlines()[-40:])
    state = f"exited with code {server.returncode}" if server.poll() is not None else "timed out"
    raise RuntimeError(f"Server {state} before {url} answered. Last log lines:\n{tail}")


def stop_server(server):
    """Stop a server started by ensure_server, and everything it started."""
    if server is not None and server.poll() is None:
        os.killpg(server.pid, signal.SIGTERM)
        try:
            server.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid, signal.SIGKILL)


Start the vLLM server in the background and wait until its API answers:

In [7]:
vllm_server = ensure_server(
    "vllm serve Qwen/Qwen3-30B-A3B"
    " --served-model-name Qwen3-30B-A3B --api-key abc-123 --port 8000"
    " --enable-auto-tool-choice --tool-call-parser hermes --trust-remote-code"
    " --gpu-memory-utilization 0.9",
    "http://localhost:8000/v1/models", "vllm_server.log", timeout=1800,
    headers={"Authorization": "Bearer abc-123"},
)

Already running: http://localhost:8000/v1/models


You can monitor the GPU utilization from a terminal while the server runs:

**Note**: For ROCm 6.4 and earlier, use the `rocm-smi` command instead.

```bash
watch amd-smi
```

After a successful launch, your server should be accepting incoming traffic through an OpenAI-compatible API. Now set some environment variables for your server to use throughout this tutorial:

In [19]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


Verify the model is available at the `BASE_URL` you just set by running the following command.

In [20]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen3-30B-A3B","object":"model","created":1794072383,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B","parent":null,"max_model_len":40960,"permission":[{"id":"modelperm-aee7dfd3794b1d18","object":"model_permission","created":1794072383,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

Congratulations, you just launched a powerful server that can serve any incoming request, allowing you to build amazing applications.

<a id="step2"></a>

## Step 2: Installing dependencies

Install the PydanticAI dependencies using this command:

In [21]:
!pip install "pydantic-ai-slim[mcp]==1.65.0"

Looking in indexes: https://pypi.org/simple, https://repo.amd.com/rocm/whl-multi-arch



<a id="step3"></a>

## Step 3: Create a simple instance of a PydanticAI agent

Start by creating a custom OpenAI-compatible endpoint for your agent. 


In [22]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

Now create an instance of the `Agent` class from `pydantic_ai`. 


In [23]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. The `pydantic_ai` framework provides multiple ways to run an `Agent` instance. To learn more, see the [PydanticAI site](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, you are running the agent in `async` mode. Define a helper function that allows you to quickly test your agent.

In [24]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


Test the agent by calling this function.

In [25]:
await run_async("What is the capital of France?")


'\n\nThe capital of France is **Paris**. It is a major city in Europe and is renowned for its cultural landmarks, such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris serves as the political, economic, and cultural center of the country.'

Now that you have the basics of an agent instance, connect it to the model you are serving with vLLM.

<a id="step4"></a>

## Step 4: Write a date/time tool for your agent

LLMs rely on their training data to respond to your prompts, so the agent you just defined would fail to answer a factual question that falls outside of its training knowledge. You can show this with an example:

In [26]:
await run_async("What’s the date today?")

'\n\nI don\'t have real-time capabilities to check the current date and time. However, you can check the date and time on your device or by searching "What is today\'s date?" in a search engine. Let me know if there\'s anything else I can help with! 😊'

It's no surprise that the model failed to answer this question. Now it's time to power-up your LLM by providing the agent with a function that can get the current date. The process by which an LLM triggers a function call is commonly referred to as "Tool Calling" or "Function Calling". In this workshop, you're going to take advantage of the `pydantic_ai` agent `Tool` package to provide the agent with appropriate tools. First, define a custom tool within this framework using the code sample below.

In [27]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


Next, provide this tool to your agent by adding the function signature of the tool to the `Agent` constructor. This notifies the LLM that the new tool exists. 

In [28]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

Now test the agent.

In [29]:
await run_async("What’s the date today?")

"\n\nToday's date is November 7, 2026 (17:27:18 UTC)."

Congratulations on building an agent with access to real-time data. 



<a id="step5"></a>

## Step 5: Replace the date/time tool with an MCP server

Now that you learned how to create a custom tool and let the agent access it, you can enhance this step using the [Model Context Protocol](https://modelcontextprotocol.io/introduction) (MCP). You can replace the custom tool with a simple MCP server that serves the agent and provides similar information.

Why should you use MCP? MCP servers provide:
- ✅ Standardized API interfaces
- 🔄 Reusability across projects
- 📦 Prebuilt functionality

To replace your custom time tool with an official MCP time server, follow these steps:

### Installing an MCP time server

Start by installing the MCP server:


In [30]:
!pip install -q mcp-server-time

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.23.1.dev1+g9ddef7117.d20260901.rocm714 requires tensorizer==2.10.1, but you have tensorizer 2.12.1 which is incompatible.


Now define the `time_server`:

In [31]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

Finally, modify your agent by removing the previously defined tool and adding the MCP server instead.

In [32]:
agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

You can now see whether the agent can use MCP to provide the correct time.

In [33]:
await run_async("What’s the date today?")

"\n\nToday's date is **Saturday, November 7, 2026**. Let me know if you need anything else!"


You have now used an official MCP server to power up your agent. In the next section, you'll learn how to turn your ideas into real working projects by using the hundreds of available free or paid MCP servers.


<a id="step6"></a>

## Step 6: Turn your agent into an Airbnb finder

As you discovered in the last section, MCP servers are very easy to use. They provide a standard way of providing LLMs with the tools they need. There are already thousands of MCP servers available for you to use. Consult one of the following MCP trackers as a reference to find out about the available servers:
- [https://github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers)
- [https://mcp.so/](https://mcp.so/)

You will use npx to launch your next server. Use the following commands to install the required dependencies:

In [ ]:
%%bash
# Install Node.js 20 via NodeSource
set -e
SUDO=""; [ "$(id -u)" = 0 ] || SUDO="sudo -E"
curl -fsSL https://deb.nodesource.com/setup_20.x | $SUDO bash -
$SUDO apt-get install -y nodejs
node --version

Verify the `npm` and `npx` installations:

In [ ]:
!node -v && npm -v && npx --version



In this part of the workshop, you're going to build an agent to help browse available Airbnbs listings to book. You can build on top of what you've done already and add an open-source Airbnb MCP server to your agent. Start by defining the Airbnb server.

In [34]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

Now update your agent.

In [35]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    toolsets=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


Finally, see if your agent can browse through the Airbnb listings.

In [36]:
await run_async("Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?")

"\n\nHere are some Airbnb options in Vancouver for your stay from **November 8 to November 11, 2026** (3 nights) for 2 adults:\n\n---\n\n### **Top Picks**\n1. **Parker Hotel and Rooftop, Deluxe Two Doubles**  \n   - **Price**: $351 for 3 nights  \n   - **Rating**: 5.0 ⭐ (6 reviews)  \n   - **Location**: Downtown Vancouver  \n   - **Features**: Hotel, rooftop access  \n\n2. **Studio Apartment in Downtown Vancouver**  \n   - **Price**: $340 for 3 nights  \n   - **Rating**: 4.9 ⭐ (83 reviews)  \n   - **Location**: Central Vancouver  \n   - **Features**: 1 bedroom, 4 beds, 1 bath  \n\n3. **Spacious - Bright - Modern & Central**  \n   - **Price**: $445 for 3 nights  \n   - **Rating**: New listing  \n   - **Location**: Downtown  \n   - **Features**: 2 bedrooms, 2 baths  \n\n---\n\n### **Other Options**\n- **Pet-Friendly Twin Room** (Starts at $297 for 3 nights)  \n- **Tranquil 1BR Condo** (Starts at $333 for 3 nights, 94 reviews)  \n- **Urban Retreat with Pool & Gym** (Starts at $436 for 3 n

When you're done, stop the vLLM server:

In [37]:
stop_server(vllm_server)



<a id="step7"></a>

## Step 7: Challenge to expand the agent

For an additional challenge, add weather integration using the MCP weather server of your choice:
1. Launch the MCP weather server
2. Add it to the list of agent tools
3. Ask the agent to suggest the best travel dates based on the weather

In [40]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Travel Assistant")

@mcp.tool()
def get_weather(city: str) -> str:
    weather = {
        "Paris": "18-22°C, mostly sunny, low rainfall",
        "London": "12-18°C, frequent showers",
        "Tokyo": "20-25°C, partly cloudy with occasional rain"
    }
    return weather.get(city, "Weather data not available")

In [41]:
get_weather("Paris")

'18-22°C, mostly sunny, low rainfall'

In [42]:
weather = get_weather("Paris")

print("Weather:", weather)

print("""
Suggested travel dates for Paris:

Choose dates with temperatures between 18-22°C,
mostly sunny conditions, and low rainfall.

These conditions are ideal for sightseeing,
walking tours, and outdoor activities.
""")

Weather: 18-22°C, mostly sunny, low rainfall

Suggested travel dates for Paris:

Choose dates with temperatures between 18-22°C,
mostly sunny conditions, and low rainfall.

These conditions are ideal for sightseeing,
walking tours, and outdoor activities.

